In [ ]:
import numpy as np
import pandas as pd
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d import proj3d

# ===== CONFIG =====
N_CLUSTERS = 3
CLUSTER_NAMES = ["Not My Favorite", "So Far, So Good", "My Favorite"]
CLUSTER_COLORS = ["red", "orange", "green"]
ELLIPSOID_SCALE = 1.6
ELLIPSOID_ALPHA = 0.10
FIGSIZE = (18, 12)
DPI = 150

STYLE_FEATURE_COLS = ["danceability", "energy", "acousticness", "valence"]
BIAS_COL = "bias_score"

# ===== 1. ข้อมูล =====
tracks = pd.DataFrame([
    {"track": "01. I Feel So Free", "danceability": 6.0, "energy": 7.0, "acousticness": 3.0, "valence": 8.0, "bias_score": 7.0},
    {"track": "02. Good for the Soul", "danceability": 7.0, "energy": 7.5, "acousticness": 2.0, "valence": 8.5, "bias_score": 8.5},
    {"track": "03. One Step Away", "danceability": 6.5, "energy": 6.5, "acousticness": 3.5, "valence": 7.0, "bias_score": 8.5},
    {"track": "04. Bring Your Love (with Sabrina Carpenter)", "danceability": 9.0, "energy": 9.0, "acousticness": 2.0, "valence": 9.0, "bias_score": 9.5},
    {"track": "05. Danceteria", "danceability": 10.0, "energy": 10.0, "acousticness": 2.0, "valence": 9.0, "bias_score": 10.0},
    {"track": "06. Read My Lips (with Feid)", "danceability": 7.0, "energy": 8.0, "acousticness": 5.0, "valence": 8.0, "bias_score": 8.0},
    {"track": "07. Everything", "danceability": 8.0, "energy": 10.0, "acousticness": 2.5, "valence": 8.0, "bias_score": 8.5},
    {"track": "08. Love Sensation", "danceability": 7.5, "energy": 8.5, "acousticness": 2.0, "valence": 9.0, "bias_score": 8.5},
    {"track": "09. Love Without Words", "danceability": 6.0, "energy": 8.0, "acousticness": 3.0, "valence": 7.0, "bias_score": 7.5},
    {"track": "10. Bizarre (with Martin Garrix)", "danceability": 8.0, "energy": 9.0, "acousticness": 2.0, "valence": 7.0, "bias_score": 9.0},
    {"track": "11. School", "danceability": 7.5, "energy": 8.5, "acousticness": 1.0, "valence": 6.0, "bias_score": 8.0},
    {"track": "12. Fragile", "danceability": 6.0, "energy": 6.0, "acousticness": 4.0, "valence": 5.0, "bias_score": 7.5},
    {"track": "13. My Sins Are My Savior (feat. Stromae)", "danceability": 6.0, "energy": 6.5, "acousticness": 4.0, "valence": 7.0, "bias_score": 8.0},
    {"track": "14. Betrayal", "danceability": 5.0, "energy": 5.0, "acousticness": 6.5, "valence": 4.5, "bias_score": 8.5},
    {"track": "15. The Test (with Lola Leon)", "danceability": 7.0, "energy": 7.5, "acousticness": 3.5, "valence": 6.0, "bias_score": 8.0},
    {"track": "16. L.E.S. Girl", "danceability": 4.0, "energy": 4.0, "acousticness": 6.0, "valence": 3.0, "bias_score": 7.0}
])

tracks["log_bias"] = np.log1p(tracks[BIAS_COL])

# ===== 2. standardize =====
all_feature_cols = STYLE_FEATURE_COLS + ["log_bias"]
bias_col_idx = all_feature_cols.index("log_bias")

X = tracks[all_feature_cols].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ===== 3. GMM fit =====
model = GaussianMixture(n_components=N_CLUSTERS, random_state=42)
raw_labels = model.fit_predict(X_scaled)
tracks["raw_cluster"] = raw_labels

cluster_means = tracks.groupby("raw_cluster")["log_bias"].mean().sort_values()
order_map = {cid: rank for rank, cid in enumerate(cluster_means.index)}
tracks["rank"] = tracks["raw_cluster"].map(order_map)
tracks["label"] = tracks["rank"].apply(lambda r: CLUSTER_NAMES[r])

# ===== 4. PCA =====
style_idx = [all_feature_cols.index(c) for c in STYLE_FEATURE_COLS]
X_style_scaled = X_scaled[:, style_idx]

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_style_scaled)
tracks["pc1"] = X_pca[:, 0]
tracks["pc2"] = X_pca[:, 1]
tracks["plot_z"] = tracks["log_bias"]

explained = pca.explained_variance_ratio_

# ===== 5. projection matrix สำหรับ ellipsoid =====
n_features = len(all_feature_cols)
W = np.zeros((3, n_features))
for row_i, style_col_pos in enumerate(style_idx):
    W[0, style_col_pos] = pca.components_[0, row_i]
    W[1, style_col_pos] = pca.components_[1, row_i]
W[2, bias_col_idx] = 1.0

bias_mean_ = scaler.mean_[bias_col_idx]
bias_std_ = scaler.scale_[bias_col_idx]

def ellipsoid_surface_custom(mean_full, cov_full, W, scale_factor, n_points=25):
    mean_proj = W @ mean_full
    cov_proj = W @ cov_full @ W.T
    mean_proj = mean_proj.copy()
    mean_proj[2] = mean_proj[2] * bias_std_ + bias_mean_
    cov_proj = cov_proj.copy()
    cov_proj[2, :] *= bias_std_
    cov_proj[:, 2] *= bias_std_

    eigvals, eigvecs = np.linalg.eigh(cov_proj)
    eigvals = np.maximum(eigvals, 1e-6)

    u = np.linspace(0, 2 * np.pi, n_points)
    v = np.linspace(0, np.pi, n_points)
    x = np.outer(np.cos(u), np.sin(v))
    y = np.outer(np.sin(u), np.sin(v))
    z = np.outer(np.ones_like(u), np.cos(v))
    sphere = np.stack([x, y, z], axis=-1)

    radii = np.sqrt(eigvals) * scale_factor
    transformed = sphere @ np.diag(radii) @ eigvecs.T
    return transformed + mean_proj

# ===== 6. เตรียมกราฟ (ต้องสร้าง ax ก่อน เพราะ proj3d ต้องใช้ ax.get_proj()) =====
fig = plt.figure(figsize=FIGSIZE, dpi=DPI)
ax = fig.add_subplot(111, projection='3d')
ax.view_init(elev=18, azim=-55)

z_min, z_max = tracks["plot_z"].min(), tracks["plot_z"].max()
z_floor = z_min - (z_max - z_min) * 0.15
ax.set_zlim(z_floor, z_max + (z_max - z_min) * 0.05)

# บังคับ set_xlim/set_ylim ให้คงที่ก่อนคำนวณ projection เพื่อไม่ให้ autoscale เปลี่ยนทีหลัง
pad_x = (tracks["pc1"].max() - tracks["pc1"].min()) * 0.3
pad_y = (tracks["pc2"].max() - tracks["pc2"].min()) * 0.3
ax.set_xlim(tracks["pc1"].min() - pad_x, tracks["pc1"].max() + pad_x)
ax.set_ylim(tracks["pc2"].min() - pad_y, tracks["pc2"].max() + pad_y)

def get_screen_coords(x, y, z):
    x_proj, y_proj, _ = proj3d.proj_transform(x, y, z, ax.get_proj())
    return np.array([x_proj, y_proj])

def dist_point_to_segment(p, a, b):
    ab = b - a
    ab_norm = np.dot(ab, ab)
    if ab_norm < 1e-12:
        return np.linalg.norm(p - a)
    t = np.clip(np.dot(p - a, ab) / ab_norm, 0.0, 1.0)
    closest = a + t * ab
    return np.linalg.norm(p - closest)

def boxes_overlap(c1, hw1, hh1, c2, hw2, hh2):
    """AABB overlap check ระหว่างกล่อง 2 ใบ (center + half-width/half-height)"""
    return (abs(c1[0] - c2[0]) < (hw1 + hw2)) and (abs(c1[1] - c2[1]) < (hh1 + hh2))

# ===== 7. คาลิเบรทขนาดกล่อง/keepout ให้เป็นสัดส่วนของพื้นที่กราฟจริง =====
# ปัญหาเดิม: proj3d.proj_transform คืนพิกัดในหน่วยที่เล็กมาก (ทั้งกราฟอาจกว้างแค่ 0.1-0.2 หน่วย)
# ถ้าใช้ค่าคงที่ตายตัว (เช่น 0.05, 0.25) จะใหญ่กว่าพื้นที่กราฟทั้งหมด ทำให้ไม่มีตำแหน่งไหน "ปลอดภัย" เลย
xlim, ylim = ax.get_xlim(), ax.get_ylim()
z_mid = tracks["plot_z"].mean()
_corners = [
    get_screen_coords(xlim[0], ylim[0], z_mid),
    get_screen_coords(xlim[1], ylim[0], z_mid),
    get_screen_coords(xlim[0], ylim[1], z_mid),
    get_screen_coords(xlim[1], ylim[1], z_mid),
]
_xs = [c[0] for c in _corners]
_ys = [c[1] for c in _corners]
SCREEN_SPAN_X = max(_xs) - min(_xs)
SCREEN_SPAN_Y = max(_ys) - min(_ys)
print(f"Screen span: x={SCREEN_SPAN_X:.4f}, y={SCREEN_SPAN_Y:.4f} (ใช้คาลิเบรทขนาดกล่อง)")

CHAR_WIDTH = SCREEN_SPAN_X * 0.0030   # ต่อตัวอักษร
BOX_HEIGHT_2D = SCREEN_SPAN_Y * 0.11
MIN_HALF_W = SCREEN_SPAN_X * 0.012
POINT_KEEPOUT = SCREEN_SPAN_Y * 0.09   # รัศมีที่ห้ามกล่อง/เส้นเข้าใกล้จุดข้อมูลใดๆ

def half_width_for(text):
    return max(MIN_HALF_W, len(text) * CHAR_WIDTH / 2)

tracks["box_hw"] = tracks["track"].apply(half_width_for)
tracks["box_hh"] = BOX_HEIGHT_2D / 2

# พิกัดหน้าจอของจุดข้อมูลจริง (คงที่ตลอด เพราะ xlim/ylim/zlim/view ถูกล็อกไว้แล้วด้านบน)
dot_screen = np.array([get_screen_coords(r["pc1"], r["pc2"], r["plot_z"]) for _, r in tracks.iterrows()])

# ===== 8. วางตำแหน่ง label ทีละจุด แบบรับประกันไม่มี "ค่า default ที่ไม่ถูกตรวจสอบ" =====
label_screen = np.zeros((len(tracks), 2))
label_pc1 = np.zeros(len(tracks))
label_pc2 = np.zeros(len(tracks))
dir_x = np.zeros(len(tracks))
dir_y = np.zeros(len(tracks))

N_ANGLES = 96  # ทุก 3.75 องศา (ละเอียดขึ้น)
radii_scales = np.arange(0.4, 12.01, 0.3)  # ขยายเพิ่มอีกสำหรับเคสจุดที่ใกล้กันมากๆ (06,07,08)

for i in range(len(tracks)):
    row = tracks.iloc[i]
    p0 = dot_screen[i]
    hw_i, hh_i = row["box_hw"], row["box_hh"]

    angles = np.linspace(0, 2 * np.pi, N_ANGLES, endpoint=False)
    angles = np.roll(angles, (i * 13) % N_ANGLES)  # กระจายเฟสเริ่มต้นให้ไม่ทุกจุดเริ่มมุมเดียวกัน

    best_candidate = None
    best_score = None  # None = ยังไม่เคยมี candidate ใดๆ เลย (fallback สุดท้ายจริงๆ)

    for radius in radii_scales:
        for angle in angles:
            off_pc1 = np.cos(angle) * radius
            off_pc2 = np.sin(angle) * radius
            test_pc1 = row["pc1"] + off_pc1
            test_pc2 = row["pc2"] + off_pc2
            test_screen = get_screen_coords(test_pc1, test_pc2, row["plot_z"])

            # เงื่อนไข 1: กล่อง label ต้องไม่ทับ "จุดข้อมูล" ใดๆ เลย (ทุกจุด รวมตัวเอง)
            box_hits_point = False
            for k in range(len(tracks)):
                dx = abs(test_screen[0] - dot_screen[k][0])
                dy = abs(test_screen[1] - dot_screen[k][1])
                if dx < (hw_i + POINT_KEEPOUT) and dy < (hh_i + POINT_KEEPOUT):
                    box_hits_point = True
                    break
            violation = 0.0
            if box_hits_point:
                violation += 100.0  # penalty หนักมาก ห้ามเกิดขึ้นถ้าเลี่ยงได้

            # เงื่อนไข 2: เส้นชี้ (จาก p0 ไป test_screen) ต้องไม่ลากผ่านใกล้จุดข้อมูลอื่น
            # keepout ปรับตัวลดลงอัตโนมัติถ้าจุด k อยู่ใกล้จุดตั้งต้น (p0) กว่ารัศมีปกติอยู่แล้ว
            # (ป้องกันเคสที่เป็นไปไม่ได้ทางเรขาคณิต เช่น 2 จุดข้อมูลอยู่ใกล้กันยิ่งกว่า keepout เอง)
            for k in range(len(tracks)):
                if k == i:
                    continue
                d = dist_point_to_segment(dot_screen[k], p0, test_screen)
                anchor_dist = np.linalg.norm(dot_screen[k] - p0)
                effective_keepout = min(POINT_KEEPOUT, 0.6 * anchor_dist)
                if d < effective_keepout:
                    violation += 100.0

            # เงื่อนไข 3: กล่อง label ต้องไม่ทับกล่อง label อื่นที่วางไปแล้ว (AABB overlap)
            for j in range(i):
                if boxes_overlap(test_screen, hw_i, hh_i,
                                  label_screen[j], tracks.iloc[j]["box_hw"], tracks.iloc[j]["box_hh"]):
                    # คะแนน penalty ตามขนาดพื้นที่ที่ทับกันจริง ไม่ใช่ threshold หยาบๆ
                    ox = (hw_i + tracks.iloc[j]["box_hw"]) - abs(test_screen[0] - label_screen[j][0])
                    oy = (hh_i + tracks.iloc[j]["box_hh"]) - abs(test_screen[1] - label_screen[j][1])
                    violation += max(ox, 0) * max(oy, 0) * 50.0

            # เก็บ candidate ที่ดีที่สุดไว้เสมอ (ไม่มี default ที่ไม่เคยถูกตรวจ)
            if best_score is None or violation < best_score:
                best_score = violation
                best_candidate = (test_pc1, test_pc2, test_screen, off_pc1, off_pc2)

            if violation == 0.0:
                break
        if best_score == 0.0:
            break

    tp1, tp2, tscreen, ox, oy = best_candidate
    label_pc1[i] = tp1
    label_pc2[i] = tp2
    label_screen[i] = tscreen
    dir_x[i] = ox
    dir_y[i] = oy

    if best_score > 0:
        print(f"WARNING: '{row['track']}' ไม่มีตำแหน่งที่ปลอดภัย 100% เลือกตัวที่ดีที่สุด (score={best_score:.1f})")

tracks["label_pc1"] = label_pc1
tracks["label_pc2"] = label_pc2
tracks["label_z"] = tracks["plot_z"]
tracks["dir_x"] = dir_x
tracks["dir_y"] = dir_y

# ===== 8b. Force-directed relaxation ในพิกัดหน้าจอ เพื่อกำจัด label-label overlap ที่เหลือ =====
def screen_jacobian(pc1, pc2, z, eps=1e-3):
    base = get_screen_coords(pc1, pc2, z)
    d_pc1 = (get_screen_coords(pc1 + eps, pc2, z) - base) / eps
    d_pc2 = (get_screen_coords(pc1, pc2 + eps, z) - base) / eps
    J = np.column_stack([d_pc1, d_pc2])
    return base, J

positions = label_screen.copy()
hw_arr = tracks["box_hw"].values
hh_arr = tracks["box_hh"].values
n = len(tracks)

N_ITERS = 600
STEP_DAMPING = 0.5
EPS = min(SCREEN_SPAN_X, SCREEN_SPAN_Y) * 0.002  # epsilon สัดส่วนกับสเกลจริง แทนค่าตายตัว 0.002 เดิม

for it in range(N_ITERS):
    moved = False
    for i in range(n):
        for j in range(i + 1, n):
            dx = positions[i][0] - positions[j][0]
            dy = positions[i][1] - positions[j][1]
            overlap_x = (hw_arr[i] + hw_arr[j]) - abs(dx)
            overlap_y = (hh_arr[i] + hh_arr[j]) - abs(dy)
            if overlap_x > 0 and overlap_y > 0:
                moved = True
                if overlap_x < overlap_y:
                    shift = (overlap_x / 2 + EPS) * STEP_DAMPING
                    sign = 1.0 if dx >= 0 else -1.0
                    positions[i][0] += sign * shift
                    positions[j][0] -= sign * shift
                else:
                    shift = (overlap_y / 2 + EPS) * STEP_DAMPING
                    sign = 1.0 if dy >= 0 else -1.0
                    positions[i][1] += sign * shift
                    positions[j][1] -= sign * shift

    for i in range(n):
        for k in range(n):
            dx = positions[i][0] - dot_screen[k][0]
            dy = positions[i][1] - dot_screen[k][1]
            ox = (hw_arr[i] + POINT_KEEPOUT) - abs(dx)
            oy = (hh_arr[i] + POINT_KEEPOUT) - abs(dy)
            if ox > 0 and oy > 0:
                moved = True
                if ox < oy:
                    shift = (ox + EPS) * STEP_DAMPING
                    sign = 1.0 if dx >= 0 else -1.0
                    positions[i][0] += sign * shift
                else:
                    shift = (oy + EPS) * STEP_DAMPING
                    sign = 1.0 if dy >= 0 else -1.0
                    positions[i][1] += sign * shift

    if not moved:
        print(f"Relaxation converged after {it} iterations (0 overlap เหลือ)")
        break
else:
    print(f"Relaxation ครบ {N_ITERS} รอบแล้ว (อาจเหลือ overlap เล็กน้อยถ้าพื้นที่ไม่พอจริงๆ)")

for i in range(n):
    pc1_cur, pc2_cur = label_pc1[i], label_pc2[i]
    target_screen = positions[i]
    for _ in range(3):
        cur_screen, J = screen_jacobian(pc1_cur, pc2_cur, tracks.iloc[i]["plot_z"])
        residual = target_screen - cur_screen
        try:
            delta = np.linalg.solve(J, residual)
        except np.linalg.LinAlgError:
            break
        pc1_cur += delta[0]
        pc2_cur += delta[1]
    label_pc1[i] = pc1_cur
    label_pc2[i] = pc2_cur
    label_screen[i] = get_screen_coords(pc1_cur, pc2_cur, tracks.iloc[i]["plot_z"])
    dir_x[i] = pc1_cur - tracks.iloc[i]["pc1"]
    dir_y[i] = pc2_cur - tracks.iloc[i]["pc2"]

tracks["label_pc1"] = label_pc1
tracks["label_pc2"] = label_pc2
tracks["dir_x"] = dir_x
tracks["dir_y"] = dir_y

# ===== 9. พล็อตจุดและ ellipsoid =====
for cid, r in order_map.items():
    subset = tracks[tracks["rank"] == r]
    color = CLUSTER_COLORS[r]

    ax.scatter(subset["pc1"], subset["pc2"], subset["plot_z"],
               color=color, s=140, edgecolor="black", linewidth=0.7,
               label=CLUSTER_NAMES[r], zorder=5)

    if subset.shape[0] > 1:
        mean_full = model.means_[cid]
        cov_full = model.covariances_[cid]
        ell = ellipsoid_surface_custom(mean_full, cov_full, W, ELLIPSOID_SCALE)
        ax.plot_surface(ell[..., 0], ell[..., 1], ell[..., 2],
                        color=color, alpha=ELLIPSOID_ALPHA, linewidth=0, zorder=1)

# ===== 10. เส้นดิ่งลงพื้น + เส้นชี้ label + กล่องข้อความ =====
for _, row in tracks.iterrows():
    color = CLUSTER_COLORS[row["rank"]]

    ax.plot([row["pc1"], row["pc1"]], [row["pc2"], row["pc2"]], [row["plot_z"], z_floor],
            color="gray", linestyle="--", linewidth=0.6, alpha=0.4, zorder=2)

    ax.plot([row["pc1"], row["label_pc1"]],
            [row["pc2"], row["label_pc2"]],
            [row["plot_z"], row["label_z"]],
            color=color, linewidth=1.1, alpha=0.9, zorder=4)

    ha = "left" if row["dir_x"] > 0.05 else "right" if row["dir_x"] < -0.05 else "center"
    va = "bottom" if row["dir_y"] > 0.05 else "top" if row["dir_y"] < -0.05 else "center"

    ax.text(row["label_pc1"], row["label_pc2"], row["label_z"], row["track"],
            fontsize=7.5, zorder=6,
            horizontalalignment=ha, verticalalignment=va,
            bbox=dict(boxstyle="round,pad=0.18", facecolor="white",
                      edgecolor=color, alpha=0.9, linewidth=0.6))

ax.set_xlabel(f"PC1 - style ({explained[0]:.0%} var)", labelpad=15, fontsize=11)
ax.set_ylabel(f"PC2 - style ({explained[1]:.0%} var)", labelpad=15, fontsize=11)
ax.set_zlabel("log(personal bias score + 1)", labelpad=20, fontsize=11)
ax.set_title("Album Track Clustering — Collision-Free Label Placement", fontsize=14, pad=20)
ax.legend(loc="upper left", bbox_to_anchor=(0.02, 0.95))

plt.subplots_adjust(left=0.02, right=0.92, top=0.92, bottom=0.05)
plt.savefig("album_clustering_fixed.png", dpi=DPI, bbox_inches="tight")
plt.show()

# ===== 11. ตรวจสอบผลลัพธ์สุดท้ายอีกครั้ง (ต้องได้ 0 ทุกตัว) =====
print("\n=== ตรวจสอบผลลัพธ์สุดท้าย ===")
box_over_point = 0
for i in range(len(tracks)):
    for k in range(len(tracks)):
        dx = abs(label_screen[i][0] - dot_screen[k][0])
        dy = abs(label_screen[i][1] - dot_screen[k][1])
        if dx < tracks.iloc[i]["box_hw"] and dy < tracks.iloc[i]["box_hh"]:
            box_over_point += 1
            print(f"BOX COVERS POINT: '{tracks.iloc[i]['track']}' covers '{tracks.iloc[k]['track']}'")
print(f"box-covers-point violations: {box_over_point}")

line_hits = 0
for i in range(len(tracks)):
    for k in range(len(tracks)):
        if k == i:
            continue
        d = dist_point_to_segment(dot_screen[k], dot_screen[i], label_screen[i])
        if d < POINT_KEEPOUT:
            line_hits += 1
            print(f"LINE HITS POINT: line of '{tracks.iloc[i]['track']}' hits dot of '{tracks.iloc[k]['track']}'")
print(f"line-hits-point violations: {line_hits}")

label_overlaps = 0
for i in range(len(tracks)):
    for j in range(i+1, len(tracks)):
        if boxes_overlap(label_screen[i], tracks.iloc[i]["box_hw"], tracks.iloc[i]["box_hh"],
                          label_screen[j], tracks.iloc[j]["box_hw"], tracks.iloc[j]["box_hh"]):
            label_overlaps += 1
            print(f"LABEL OVERLAP: '{tracks.iloc[i]['track']}' <-> '{tracks.iloc[j]['track']}'")
print(f"label-label overlaps: {label_overlaps}")